# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore, load, and process the FAIR² mass spectrometry dataset using the `mlcroissant` library. The dataset is defined by a Croissant JSON-LD schema and contains rich, structured tables suitable for data science workflows.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We load metadata and records from the Croissant-described dataset using `mlcroissant`. The schema URL provides all necessary structure for data access.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset loaded: {metadata.name}\n\nDescription: {metadata.description}\n\nCite as: {metadata.citeAs if hasattr(metadata, 'citeAs') else 'N/A'}")

## 2. Data Overview
Review the available record sets, their `@id`s, and their fields/columns.

We use the metadata object to enumerate all record sets, fields, and their identifiers for reference in further steps.

In [ ]:
# List all record sets in the dataset
record_sets = []
print("Record sets (by @id) in this dataset:")
for rs in metadata.recordSets:
    print(f"- @id: {rs.id} | name: {rs.name}")
    record_sets.append(rs.id)

    # List associated fields
    if hasattr(rs, 'fields'):
        print("  Fields and their column @id's:")
        for f in rs.fields:
            col_ids = [col.id for col in f.columns] if hasattr(f, 'columns') else []
            print(f"    - field @id: {f.id} | field name: {f.name} | column @ids: {col_ids}")
    print("")

## 3. Data Extraction
Here we demonstrate how to load data from a chosen record set into a DataFrame for analysis. Make sure to use the exact record set and field `@id`s as shown above.

In [ ]:
# Select a primary record set for the main table (adjust as needed based on previous cell output):
if record_sets:
    selected_record_set = record_sets[0]
    print(f"Selected record set: {selected_record_set}")
else:
    raise ValueError("No record sets found.")

# Extract all available record sets into DataFrames
dataframes = {}
for rs_id in record_sets:
    print(f"Loading record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Columns in '{rs_id}': {dataframes[rs_id].columns.tolist()}")
        print(dataframes[rs_id].head(2))
    else:
        print(f"No records found for {rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Now let's perform basic filtering, normalization, and grouping on the data using the record set and field `@id` references.

First, we pick a numeric column from the selected record set for demonstration.

In [ ]:
# Demo EDA on the main data table
from pandas.api.types import is_numeric_dtype
df = dataframes[selected_record_set]

# Display all columns with their types
print("Column data types:")
print(df.dtypes)

# Find a numeric field for analysis
numeric_cols = [c for c in df.columns if is_numeric_dtype(df[c])]
if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Selected numeric field (for filtering/normalization): {numeric_field}")
else:
    raise ValueError("No numeric fields found in the record set.")

# Filter: only keep records where numeric_field > threshold
threshold = df[numeric_field].quantile(0.5)  # Use median as example threshold
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
print(filtered_df.head(5))

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a likely categorical column
cat_cols = [c for c in df.columns if df[c].dtype == object and df[c].nunique() < df.shape[0]//2]
if cat_cols:
    group_field = cat_cols[0]
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
    print(grouped_df.head())
else:
    print("No suitable categorical column for grouping found.")

## 5. Visualization
Visualize data distributions and relationships, making use of well-known libraries such as matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field], kde=True, bins=30)
plt.title(f"Distribution of '{numeric_field}'")
plt.xlabel(numeric_field)
plt.tight_layout()
plt.show()

# Boxplot grouped by selected group_field
if 'group_field' in locals():
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"'{numeric_field}' grouped by '{group_field}'")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated:

- How to load a Croissant-described mass spectrometry dataset with `mlcroissant`.
- How to enumerate and reference record sets and fields by `@id`.
- How to extract and analyze data with pandas, including filtering, normalization, and grouping.
- How to visualize distributions and relationships within the data.

Please explore the full data dictionary and schema for deeper biological or clinical insights using the rich annotation and metadata provided!